# Figure 1 — multiscale luminance geometry (no Van Gogh)

This notebook regenerates the revised **Figure 1** using three objectively selected Post-Impressionist paintings from different artists:

- **Low geometry:** Anita Malfatti, *Fernanda de Castro* (1922)
- **Intermediate geometry:** Amrita Sher-Gil, *Tribal Women* (1938)
- **High geometry:** Abraham Manievich, *The Yellow House*

The paintings are selected near the 5th, 50th and 95th percentiles of `geom__curv__kappa_ref_s2p0_grad_weighted_abs` within the Post-Impressionism subset. The multiscale curvature panel uses *Tribal Women* at $\sigma_{ref}=1,2,4,8$.


In [ ]:
!pip -q install scipy pandas matplotlib pillow

from pathlib import Path
import io, zipfile, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.patches import FancyBboxPatch
from PIL import Image
from scipy.ndimage import gaussian_filter
from google.colab import files

BASE = Path('/content')
OUT = BASE / 'paper_figures_main'
OUT.mkdir(exist_ok=True)
print('Ready.')


## Upload inputs
Upload these four files when prompted:

1. `painting_geometry_phase4_artbench_pilot.zip`
2. `anita-malfatti_fernanda-de-castro-1922.jpg`
3. `amrita-sher-gil_tribal-women-1938.jpg`
4. `abraham-manievich_the-yellow-house.jpg`


In [ ]:
uploaded = files.upload()
for name, payload in uploaded.items():
    (BASE / name).write_bytes(payload)
print('Uploaded:', list(uploaded))


In [ ]:
PHASE4_ZIP = BASE / 'painting_geometry_phase4_artbench_pilot.zip'
with zipfile.ZipFile(PHASE4_ZIP) as zf:
    members = zf.namelist()
    matches = [m for m in members if m.endswith('artbench_pilot_features.csv')]
    if not matches:
        raise FileNotFoundError('artbench_pilot_features.csv not found inside Phase IV ZIP')
    with zf.open(matches[0]) as f:
        features = pd.read_csv(f)

DESC = 'geom__curv__kappa_ref_s2p0_grad_weighted_abs'
SELECTION = [
    dict(role='Low geometry', style='post_impressionism', artist='anita-malfatti', artist_label='Anita Malfatti', title='Fernanda de Castro', year='1922', filename='anita-malfatti_fernanda-de-castro-1922.jpg', expected=0.252172),
    dict(role='Intermediate geometry', style='post_impressionism', artist='amrita-sher-gil', artist_label='Amrita Sher-Gil', title='Tribal Women', year='1938', filename='amrita-sher-gil_tribal-women-1938.jpg', expected=0.324556),
    dict(role='High geometry', style='post_impressionism', artist='abraham-manievich', artist_label='Abraham Manievich', title='The Yellow House', year='', filename='abraham-manievich_the-yellow-house.jpg', expected=0.403162),
]

rows=[]
for s in SELECTION:
    hit = features[(features['style'].astype(str)==s['style']) & (features['artist'].astype(str)==s['artist']) & (features['filename'].astype(str)==s['filename'])]
    if len(hit)!=1:
        raise RuntimeError(f"Expected one row for {s['filename']}, found {len(hit)}")
    value=float(hit.iloc[0][DESC])
    rows.append({**s,'value':value})

check = pd.DataFrame(rows)[['role','artist_label','title','year','filename','value','expected']]
display(check)


In [ ]:
def resize_long_side(img, long_side=256):
    w,h=img.size
    scale=float(long_side)/max(w,h)
    return img.resize((max(1,round(w*scale)),max(1,round(h*scale))), Image.Resampling.LANCZOS)

def luminance_bt601(rgb):
    a=np.asarray(rgb,dtype=float)
    y=0.299*a[...,0]+0.587*a[...,1]+0.114*a[...,2]
    y-=y.min(); d=y.max()-y.min()
    return np.zeros_like(y) if d<=0 else y/d

def curvature_map(I, sigma_ref, long_side=256, reference_long_side=512, eps=1e-12, grad_quantile=.20):
    sigma_px=float(sigma_ref)*float(long_side)/float(reference_long_side)
    kw=dict(sigma=sigma_px,mode='reflect',truncate=3.0)
    Ix=gaussian_filter(I,order=(0,1),**kw); Iy=gaussian_filter(I,order=(1,0),**kw)
    Ixx=gaussian_filter(I,order=(0,2),**kw); Iyy=gaussian_filter(I,order=(2,0),**kw); Ixy=gaussian_filter(I,order=(1,1),**kw)
    g2=Ix*Ix+Iy*Iy; g=np.sqrt(g2)
    k=(Ixx*Iy*Iy-2*Ix*Iy*Ixy+Iyy*Ix*Ix)/np.power(g2+eps*eps,1.5)
    finite=np.isfinite(k)&np.isfinite(g); pos=g[finite&(g>0)]
    thr=np.quantile(pos,grad_quantile) if pos.size else 0.0
    valid=finite&(g>=thr)
    return sigma_px*k, valid

rgbs=[]; lums=[]
for s in rows:
    p=BASE/s['filename']
    if not p.exists(): raise FileNotFoundError(p)
    rgb=resize_long_side(Image.open(p).convert('RGB'),256)
    rgbs.append(np.asarray(rgb)); lums.append(luminance_bt601(rgb))
print('Images loaded.')


In [ ]:
fig=plt.figure(figsize=(13.8,10.2))
gs=gridspec.GridSpec(3,6,figure=fig,height_ratios=[1.0,.92,1.0],hspace=.34,wspace=.16)

def plabel(ax,s):
    ax.text(-.06,1.04,s,transform=ax.transAxes,fontsize=15,fontweight='bold',va='bottom')

# a: three paintings
for j,(s,rgb) in enumerate(zip(rows,rgbs)):
    ax=fig.add_subplot(gs[0,2*j:2*j+2])
    if j==0: plabel(ax,'a')
    ax.imshow(rgb); ax.axis('off')
    yr=f" ({s['year']})" if s['year'] else ''
    ax.set_title(f"{s['role']}\n{s['artist_label']}, {s['title']}{yr}\n$G_{{\sigma=2}}={s['value']:.3f}$",fontsize=9.2,pad=5)

# b: luminance + contours
for j,(s,I) in enumerate(zip(rows,lums)):
    ax=fig.add_subplot(gs[1,2*j:2*j+2])
    if j==0: plabel(ax,'b')
    ax.imshow(I,cmap='gray',vmin=0,vmax=1)
    lev=np.unique(np.quantile(I,[.15,.30,.45,.60,.75,.90]))
    ax.contour(I,levels=lev,colors='white',linewidths=.45,alpha=.85)
    ax.axis('off'); ax.set_title(f"Iso-luminance contours — {s['role'].replace(' geometry','').lower()}",fontsize=8.7)

# c: scale maps for Tribal Women
sub=gridspec.GridSpecFromSubplotSpec(1,4,subplot_spec=gs[2,0:4],wspace=.08)
I=lums[1]
for j,sigma in enumerate([1,2,4,8]):
    ax=fig.add_subplot(sub[0,j])
    if j==0: plabel(ax,'c')
    k,valid=curvature_map(I,sigma_ref=sigma,long_side=max(I.shape))
    vals=np.abs(k[valid]); clip=np.percentile(vals,99) if vals.size else np.nanpercentile(np.abs(k),99)
    ax.imshow(k,cmap='coolwarm',vmin=-clip,vmax=clip); ax.axis('off'); ax.set_title(rf'$\sigma_{{ref}}={sigma}$')

# d: summaries
ax=fig.add_subplot(gs[2,4:6]); plabel(ax,'d'); ax.axis('off')
ax.text(.02,.96,'Image-level summaries',transform=ax.transAxes,ha='left',va='top',fontsize=10.5,fontweight='bold')
summaries=[r'median / mean $|\tilde{\kappa}|$',r'$Q_{75},Q_{90},Q_{95}$ of $|\tilde{\kappa}|$',r'signed median + positive fraction',r'entropy of signed curvature',r'gradient-weighted $|\tilde{\kappa}|$']
for i,t in enumerate(summaries):
    y=.78-i*.145
    ax.add_patch(FancyBboxPatch((.04,y-.055),.90,.095,boxstyle='round,pad=.012,rounding_size=.018',transform=ax.transAxes,fc='white',ec='.75',lw=1))
    ax.text(.08,y,t,transform=ax.transAxes,va='center',ha='left',fontsize=8.6)

fig.suptitle('Figure 1. From painting to multiscale luminance geometry',x=.02,ha='left',fontsize=14.5,fontweight='bold',y=.99)
fig.text(.02,.955,'Within one style category, paintings occupy markedly different positions in the level-set geometry measured at an intermediate spatial scale.',fontsize=9,color='.35')

for ext in ['png','pdf','svg']:
    p=OUT/f'Figure1_multiscale_luminance_geometry.{ext}'
    fig.savefig(p,dpi=300 if ext=='png' else None,bbox_inches='tight')
    print(p)
plt.show()
